# 09 - Comparison, ideal case

The five methods of block 0 against the same ground truth, on one grid, with one set of
metrics. 19 and 29 reuse what is defined here, so the definitions matter more than the
numbers.

## Two problems this notebook has to solve first

**The methods do not share a pixel grid.** 01, 02 and 05 reconstruct on the ptychographic
grid, $\lambda/(2\theta_{\max}) \approx 0.19\ \mathrm{\AA}$, set by the 52.47 mrad
collection angle. 03 lands on the scan grid, 0.36388 Å. The ground truth is on the
simulation grid, 0.1 Å. Nothing can be compared until they are resampled onto one
common grid, which is what `to_common_grid` does - each result carries its own
`pixel_size_ang`, recorded at save time rather than inferred here.

**A peak is not a metric.** Notebook 10 used a contrast whose numerator was
$\max|\Delta\varphi|$, and because a maximum over many pixels is an extreme-value
statistic that grows with noise, it ranked a *drifted* result above the ideal one. Every
metric below therefore has an **integrated** numerator - an inner product against the
truth, which is the matched filter for a known signal shape:

| metric | definition | what it says |
|---|---|---|
| `scale` | $a = \langle r, t\rangle / \langle t, t\rangle$ | how much of the true amplitude came back; 1.0 is perfect |
| `rho` | Pearson $\rho(r, t)$ | shape fidelity, independent of scale |
| `nrmse` | $\lVert r - t\rVert / \lVert t\rVert$ | total error including the scale miss |
| `snr` | $a\lVert t\rVert / \mathrm{std}(r - at)$ | detection strength: matched-filter response over residual |

`scale` and `rho` separate two different failures that a single number hides - a method
that recovers the right shape at a tenth of the amplitude is not the same as one that
gets the amplitude right and the shape wrong.

**04 is excluded from this table on purpose.** It returns a position, not an image, so its
figure of merit is localisation error in Å. Mixing the two would make the table
meaningless. It gets its own row at the end.

A global constant is removed from every $\Delta\varphi$ before comparison: 01 subtracts
two independently reconstructed phases, each with its own arbitrary gauge, so a constant
offset carries no information.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
RESULTS_DIR = NOTEBOOK_DIR / "results"
sys.path.insert(0, str(NOTEBOOK_DIR.parent))

DOSE_KEYS = ["inf", "1e6", "1e5"]
METHODS = {
    # 01 produces two, one per scan configuration: it is a sequential two-scan
    # experiment generated inside notebook 01, not a de-interlacing of 00's cube.
    "01_step1x": "01 subtract, 1x step",
    "01_step2x": "01 subtract, 2x step",
    "02_unreg": "02 joint, unreg",
    "02_reg": "02 joint, reg",
    "03": "03 SSB",
    "05": "05 linearized",
}

with open(DATA_DIR / "ideal_geometry.json") as handle:
    META = json.load(handle)
delta_phi_true = np.load(DATA_DIR / "ideal_truth.npz")["delta_phi_true"].astype(np.float64)

object_dx_ang = META["cell"]["dx_ang"]
n_object = META["cell"]["n_pixels"]
object_x_ang = (np.arange(n_object) - n_object // 2) * object_dx_ang
blob_centre = np.array(META["object"]["blob_centre_ang"])
scan_step_ang = META["scan"]["step_ang"]
n_scan = META["scan"]["n_positions_per_side"]
scan_axis = (np.arange(n_scan) - (n_scan - 1) / 2) * scan_step_ang

print(f"truth {delta_phi_true.shape} at {object_dx_ang:.5f} A/px, "
      f"+/-{object_x_ang[-1]:.2f} A")
print(f"truth peak |delta phi| = {np.abs(delta_phi_true).max():.5f} rad")
print(f"scan: {len(scan_axis)}^2 at {scan_step_ang} A, "
      f"{scan_axis[0]:+.2f} .. {scan_axis[-1]:+.2f} A")

In [ ]:
def centred_axis(n, pixel):
    """Physical axis of an array that is centred on the scan centre."""
    return (np.arange(n) - n / 2 + 0.5) * pixel


def load_result(key, dose_key):
    path = RESULTS_DIR / f"{key}_dose_{dose_key}.npz"
    if not path.exists():
        return None
    with np.load(path) as handle:
        out = {k: handle[k] for k in handle.files}
    return out


def to_common_grid(array, pixel, axis_common):
    """Resample a centred array onto the common physical axis."""
    axis = centred_axis(array.shape[0], pixel)
    coords = (axis_common - axis[0]) / pixel
    gy, gx = np.meshgrid(coords, coords, indexing="ij")
    return map_coordinates(array.astype(np.float64), np.vstack([gy.ravel(), gx.ravel()]),
                           order=1, mode="nearest").reshape(gy.shape)


# survey what every method produced, and how big its field of view is
inventory = {}
for key in METHODS:
    for dose_key in DOSE_KEYS:
        res = load_result(key, dose_key)
        if res is None:
            print(f"  MISSING {key} dose {dose_key}")
            continue
        arr = res["delta_phi"]
        px = res.get("pixel_size_ang")
        px = float(np.atleast_1d(px)[0]) if px is not None else np.nan
        inventory[(key, dose_key)] = dict(arr=arr, px=px)
        if dose_key == DOSE_KEYS[0]:
            print(f"  {METHODS[key]:18s} {str(arr.shape):12s} "
                  f"{px:.5f} A/px  half-FOV {arr.shape[0]/2*px:6.2f} A")

assert inventory, "no results found - run 01-05 first"
half_fov = min(v["arr"].shape[0] / 2 * v["px"] for v in inventory.values())
half_fov = min(half_fov, object_x_ang[-1]) * 0.98
AXIS_COMMON = np.arange(-half_fov, half_fov + 1e-9, object_dx_ang)
print(f"\ncommon grid: {len(AXIS_COMMON)}^2 at {object_dx_ang:.5f} A/px, "
      f"+/-{half_fov:.2f} A  (the smallest field of view any method produced)")

TRUTH = to_common_grid(delta_phi_true, object_dx_ang, AXIS_COMMON)
TRUTH = TRUTH - TRUTH.mean()
print(f"truth on common grid: peak {np.abs(TRUTH).max():.5f} rad, "
      f"||t|| = {np.linalg.norm(TRUTH):.4f}")

In [ ]:
def compare(recovered, truth=None):
    """Matched-filter metrics. Every numerator is an inner product, never a peak."""
    t = TRUTH if truth is None else truth
    r = recovered - recovered.mean()           # the global phase gauge carries nothing
    tt = float(np.vdot(t, t).real)
    scale = float(np.vdot(t, r).real) / tt
    residual = r - scale * t
    denom = residual.std()
    return dict(
        scale=scale,
        rho=float(np.corrcoef(r.ravel(), t.ravel())[0, 1]),
        nrmse=float(np.linalg.norm(r - t) / np.linalg.norm(t)),
        snr=float(abs(scale) * np.linalg.norm(t) / denom) if denom > 0 else np.inf,
    )


rows = []
for key, label in METHODS.items():
    for dose_key in DOSE_KEYS:
        item = inventory.get((key, dose_key))
        if item is None:
            continue
        grid = to_common_grid(item["arr"], item["px"], AXIS_COMMON)
        m = compare(grid)
        m.update(method=key, label=label, dose=dose_key)
        rows.append(m)
        inventory[(key, dose_key)]["grid"] = grid

print(f"{'method':20s} {'dose':>5s} | {'scale':>8s} {'rho':>8s} {'nrmse':>8s} {'snr':>9s}")
print("-" * 68)
last = None
for r in rows:
    if last is not None and r["label"] != last:
        print("-" * 68)
    print(f"{r['label']:20s} {r['dose']:>5s} | {r['scale']:8.4f} {r['rho']:8.4f} "
          f"{r['nrmse']:8.4f} {r['snr']:9.3f}")
    last = r["label"]
print("-" * 68)
print("scale 1.0 = full amplitude recovered; rho 1.0 = perfect shape; "
      "nrmse 0 = perfect; snr higher is better")

In [ ]:
ncol = 1 + len(METHODS)
fig, axes = plt.subplots(len(DOSE_KEYS), ncol,
                         figsize=(3.5 * ncol, 3.7 * len(DOSE_KEYS)),
                         constrained_layout=True, squeeze=False)
ext = [AXIS_COMMON[0], AXIS_COMMON[-1]] * 2
vmax = float(np.abs(TRUTH).max())

for row, dose_key in enumerate(DOSE_KEYS):
    ax = axes[row][0]
    im = ax.imshow(TRUTH.T, extent=ext, origin="lower", cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax)
    ax.set_title(r"ground truth $\Delta\varphi$" if row == 0 else "", fontsize=10)
    ax.set_ylabel(f"dose {dose_key}\n" + r"y [$\AA$]")
    fig.colorbar(im, ax=ax, shrink=0.78)

    for col, (key, label) in enumerate(METHODS.items(), start=1):
        ax = axes[row][col]
        item = inventory.get((key, dose_key))
        if item is None or "grid" not in item:
            ax.text(0.5, 0.5, "missing", ha="center", va="center",
                    transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([])
            continue
        g = item["grid"] - item["grid"].mean()
        lim = float(np.abs(g).max())
        im = ax.imshow(g.T, extent=ext, origin="lower", cmap="RdBu_r",
                       vmin=-lim, vmax=lim)
        m = next(r for r in rows if r["method"] == key and r["dose"] == dose_key)
        ax.set_title(f"{label}\nscale {m['scale']:.3f}  "
                     r"$\rho$ " + f"{m['rho']:.3f}", fontsize=9.5)
        fig.colorbar(im, ax=ax, shrink=0.78)
    for ax in axes[row]:
        ax.set_xlabel(r"x [$\AA$]")

fig.suptitle("09 - recovered $\\Delta\\varphi$, each panel on its own scale", fontsize=13)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19.0, 4.2), constrained_layout=True)
xpos = np.arange(len(DOSE_KEYS))
colours = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, (field, nice, logy) in zip(axes, [
        ("scale", "scale  (1.0 = full amplitude)", False),
        ("rho", r"$\rho$  (shape fidelity)", False),
        ("nrmse", "NRMSE  (lower is better)", True),
        ("snr", "SNR  (matched filter / residual)", True)]):
    for i, (key, label) in enumerate(METHODS.items()):
        vals = [next((r[field] for r in rows
                      if r["method"] == key and r["dose"] == d), np.nan)
                for d in DOSE_KEYS]
        ax.plot(xpos, vals, "o-", lw=2.0, ms=7, color=colours[i], label=label)
    if field == "scale":
        ax.axhline(1.0, color="#444", ls="--", lw=1.0)
    if field == "rho":
        ax.axhline(0.0, color="#444", ls="-", lw=0.8)
    if logy:
        ax.set_yscale("log")
    ax.set_xticks(xpos); ax.set_xticklabels(DOSE_KEYS)
    ax.set_xlabel(r"dose [e/$\AA^2$]"); ax.set_title(nice, fontsize=11)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=9, loc="best")
fig.suptitle("09 - metrics against dose", fontsize=13)
plt.show()

In [ ]:
j0 = int(np.argmin(np.abs(AXIS_COMMON - blob_centre[1])))
fig, axes = plt.subplots(1, len(DOSE_KEYS), figsize=(6.2 * len(DOSE_KEYS), 4.4),
                         constrained_layout=True, squeeze=False)
for col, dose_key in enumerate(DOSE_KEYS):
    ax = axes[0][col]
    ax.plot(AXIS_COMMON, TRUTH[:, j0], lw=3.0, color="#111111",
            label="truth", zorder=5)
    for i, (key, label) in enumerate(METHODS.items()):
        item = inventory.get((key, dose_key))
        if item is None or "grid" not in item:
            continue
        g = item["grid"] - item["grid"].mean()
        m = next(r for r in rows if r["method"] == key and r["dose"] == dose_key)
        # rescaled by the fitted amplitude, so shapes can be compared at all
        ax.plot(AXIS_COMMON, g[:, j0] / m["scale"] if m["scale"] != 0 else g[:, j0],
                lw=1.4, color=colours[i], alpha=0.9, label=f"{label} /{m['scale']:.2f}")
    ax.axvline(blob_centre[0], color="#888", ls=":", lw=1.0)
    ax.set_title(f"dose {dose_key}: cut at y = {AXIS_COMMON[j0]:+.2f} $\\AA$\n"
                 "each curve divided by its own fitted scale", fontsize=10)
    ax.set_xlabel(r"x [$\AA$]"); ax.set_ylabel(r"$\Delta\varphi$ [rad]")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()

In [ ]:
print("04 - STFT localisation, scored separately (a position, not an image)\n")
print(f"{'dose':>5s} | {'peak [A]':>20s} {'truth [A]':>20s} {'error [A]':>10s}")
print("-" * 62)
stft_rows = []
for dose_key in DOSE_KEYS:
    path = RESULTS_DIR / f"04_dose_{dose_key}.json"
    if not path.exists():
        print(f"{dose_key:>5s} |  MISSING")
        continue
    with open(path) as handle:
        rec = json.load(handle)
    found = rec["peak_xy_ang"]; truth = rec["truth_xy_ang"]
    err = float(rec["localisation_error_ang"])
    stft_rows.append(dict(dose=dose_key, error=err))
    print(f"{dose_key:>5s} | ({found[0]:+8.2f},{found[1]:+8.2f}) "
          f"({truth[0]:+8.2f},{truth[1]:+8.2f}) {err:10.2f}")
print("-" * 62)
print(f"for scale: the excitation taper is {9.0} A in radius and one scan step is "
      f"{scan_step_ang} A")

In [ ]:
summary = {
    "grid": {
        "half_fov_ang": round(float(half_fov), 4),
        "dx_ang": round(float(object_dx_ang), 6),
        "n_pixels": int(len(AXIS_COMMON)),
        "note": "the smallest field of view any method produced, at the truth's sampling",
    },
    "metric_definitions": {
        "scale": "<r,t>/<t,t>; 1.0 = full amplitude recovered",
        "rho": "Pearson correlation; shape fidelity, scale-free",
        "nrmse": "||r - t|| / ||t||",
        "snr": "|scale| * ||t|| / std(r - scale*t); matched filter over residual",
        "gauge": "a global constant is removed from every delta phi before comparison",
    },
    "image_methods": [
        {k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()}
        for r in rows
    ],
    "localisation_04": [
        {"dose": s["dose"], "error_ang": round(s["error"], 4)} for s in stft_rows
    ],
}
with open(RESULTS_DIR / "09_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)
print("wrote 09_summary.json")
print(f"{len(rows)} image-metric rows, {len(stft_rows)} localisation rows")

## How to read this

`scale` and `rho` first, and in that order. A method with $\rho$ near 1 and a small
`scale` has found the right excitation and under-weighted it, which is a calibration
problem. A method with a good `scale` and poor $\rho$ has the energy in the wrong place,
which is not.

`nrmse` is the only metric that penalises both at once, so it is the single number if one
is wanted - but it hides which of the two failed, which is why it is not the only one
here.

The metrics are defined in `compare()` and the grid handling in `to_common_grid()`.
Note that `01` appears twice. It is the only method here that does not consume notebook
00's interlaced cube - it builds two sequential scans of its own at equal total dose, on
grids that are not registered to each other, and `step2x` is the one to quote against the
rest because it reconstructs on the same 0.3282 Å grid that de-interlacing 00's cube
gives.

**19 and 29 should import this notebook's definitions rather than restate them**, so that
the ideal, drifted and real-probe blocks are scored identically. If anything here changes,
it changes for all three.